In [ ]:
import argparse
import csv
import os
import random
import sys
from pathlib import Path

_CODE_ROOT = Path(__file__).resolve().parents[1]
if str(_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(_CODE_ROOT))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gymnasium as gym
import swanlab

from device_utils import describe_device, print_device_report, resolve_torch_device

### ActorCritic 网络

|参数|来源|CartPole 中含义|网络中的体现|
|---|---|---|---|
|`obs_dim=4`|环境观测空间|小车位置、速度、杆角度、角速度|输入层 `Linear(4, 64)`|
|`act_dim=2`|环境动作空间|向左推、向右推|Actor 输出层 `Linear(64, 2)`|
|`hidden=64`|人为超参数|每个隐藏层 64 个神经元|`Linear(4,64)`、`Linear(64,64)`|

`Tanh` 与 `Sigmoid`多了一个零中心化输出，有助于数据的稳定与收敛。

---

`_init_weight` 线性层进行权重初始化 

遍历 Actor / Critic 里的每一层：
- 如果是线性层 `nn.Linear`，就用正交初始化初始化权重。
- `gain=np.sqrt(2)` 是为了配合后面的 `Tanh` 激活函数，保持前向传播时信号方差比较稳定。
- 偏置 `bias` 全部初始化为 0。

---

`self.actor[-1]` 是 Actor 的最后一层，也就是输出动作 logits 的那一层 \
权重的 gain 设为 0.01，偏置设为 0 \
Critic 同理

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim=4, act_dim=2, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, act_dim),
        )
        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
        self._init_weights()

    def _init_weight(self):
        for module in self.actor:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
                nn.init.constant_(module.bias, 0)
        for module in self.critic:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
                nn.init.constant_(module.bias, 0)
        nn.init.orthogonal_(self.actor[-1].weight, gain=0.01)
        nn.init.constant_(self.actor[-1].bias, 0)
        nn.init.orthogonal_(self.critic[-1].weight, gain=1.0)
        nn.init.constant_(self.critic[-1].bias, 0)

    def forward(self, x):
        logits = self.actor(x)
        value = self.critic(x)
        return logits, value.squeeze(-1)

    def get_action(self, obs, deterministic=False)
        logits, value = self.forward(obs)
        dist = torch.distributions.Categorical(logits=logits)
        if deterministic:
            action = logits.argmax(dim=-1)
        else:
            action = dist.sample()
        log_prob = dist.log_prob(action)
        return action, log_prob, value

### 收集轨迹 Rollout

每个 transition 记录：

- obs, action, log_prob：采样的数据
- value：当前状态的价值 V(s)
- reward：即时奖励
- terminated vs truncated：关键区分
  - terminated：杆子倒了——未来奖励为0
  - truncated：达到500步上限——任务还没失败，需要用 V(s') bootstrap

In [ ]:
for _ in range(num_steps):
    obs_tensor = torch.FloatTensor(obs)
    action, log_prob, value = model.get_action(obs_tensor)
    next_obs, reward, terminated, truncated, _ = env.step(action)
    
    # 关键：记录 next_value 用于 GAE
    if terminated:
        next_value = 0.0        # 真正结束，没有未来
    else:
        next_value = model(next_obs)  # 截断/未结束，需要bootstrap

### GAE 优势估计

三个关键参数：

- gamma=0.99：折扣因子，越接近1越重视长期奖励
- lam=0.95：GAE的λ，控制偏差-方差权衡
    - λ=0：只看一步TD，低方差但高偏差
    - λ=1：等价于蒙特卡洛，无偏但高方差
    - 0.95：折中，既稳定又有一定前瞻性

GAE公式直觉：
```
δₜ = rₜ + γV(sₜ₊₁) - V(sₜ)        # "实际比预期多多少"
Aₜ = δₜ + γλ(1-done) × Aₜ₊₁         # 把未来的优势也折现过来
```

返回值处理：
```
returns = advantages + values          # 未归一化的return target
advantages = (advantages - mean) / std # 只对advantage做标准化
```
Critic学习的是真实的 return target（不归一化），但策略梯度需要标准化后的advantage来稳定训练。

In [ ]:
for step in reversed(range(len(transitions))):
    delta = reward + gamma * next_value - value    # TD误差
    gae = delta + gamma * lam * (1 - episode_end) * gae   # GAE递推

### PPO 更新

为什么需要裁剪？

- 如果 `ratio` 偏离1太远（策略更新太剧烈），PPO会拒绝更新
- `clip(ratio, 0.8, 1.2)` 意味着：新策略概率不能偏离旧策略超过20%
- `min(surr1, surr2)` 是悲观取值——如果更新方向有利但太大，就限制住

Value Loss + Entropy：
```
value_loss = (values - returns)²            # Critic回归损失
entropy = dist.entropy().mean()             # 鼓励探索的熵奖励
loss = policy_loss + 0.5*value_loss - 0.0*entropy
```
- 这里 `entropy` 系数设为0（`-0.0*entropy`），说明CartPole足够简单不需要熵奖励
- 梯度裁剪 `clip_grad_norm_(0.5)` 防止梯度爆炸

In [ ]:
# 核心公式
ratio = exp(new_log_prob - old_log_prob)        # 新旧策略的概率比
surr1 = ratio * advantages                       # 未裁剪的目标
surr2 = clip(ratio, 1-ε, 1+ε) * advantages     # 裁剪后的目标
policy_loss = -min(surr1, surr2).mean()          # 取较小值，保守更新

### 主训练循环

学习率线性衰减：从 3e-4 到 0，训练后期更稳定，避免在最优解附近震荡。

训练完成后：

- 跑20个episode的评估（deterministic=True，取概率最大的动作）
- 保存模型权重到 `.pth` 文件
- 可选弹出GUI动画窗口

In [ ]:
for iteration in range(40):
    # 1. 采样
    transitions = collect_rollout(model, env, ...)
    
    # 2. 计算GAE
    advantages, returns = compute_gae(transitions)
    
    # 3. 学习率衰减
    lr = 3e-4 * (1 - iteration/40)
    
    # 4. PPO更新（10个epoch × mini-batch）
    metrics = ppo_update(model, optimizer, transitions, ...)
    
    # 5. 记录日志
    swanlab.log({...})

### 整体流程总结

```
采样2048步 → 计算GAE优势 → PPO更新(10 epochs × mini-batch) → 循环40次
     ↑                                                      |
     └──────────────────────────────────────────────────────┘
```